# MIMIC-IV Wide Table — Per-Column Statistics

Cell-by-cell exploratory stats for the wide table (`D:/ESILV_S2/Intern/build_mimic/mimiciv/output/mimic4_wide.parquet`).
Run cells top to bottom; the function-definition + single-column test cells can be re-run on other columns for spot checks before running the full batch in section 6.

## 0. Setup

In [ ]:
# Fix for khiops_env on Windows: the conda env's DLL directories are not on PATH
# by default when VS Code launches this kernel, which crashes the kernel process
# as soon as duckdb/pandas (and their C extensions) are imported. Add them first.
import os

_env_root = r"D:\Software\miniconda\envs\khiops_env"
_extra_paths = [
    _env_root,
    os.path.join(_env_root, "Library", "mingw-w64", "bin"),
    os.path.join(_env_root, "Library", "usr", "bin"),
    os.path.join(_env_root, "Library", "bin"),
    os.path.join(_env_root, "Scripts"),
    os.path.join(_env_root, "bin"),
]
os.environ["PATH"] = os.pathsep.join(_extra_paths) + os.pathsep + os.environ["PATH"]

# Connect to DuckDB and set the paths / column names for this table
import duckdb
import pandas as pd

PARQUET_PATH = "D:/ESILV_S2/Intern/build_mimic/mimiciv/output/mimic4_wide.parquet"
SUBJECT_COL = "subject_id"
HADM_COL = "hadm_id"
STAY_COL = "stay_id"
HR_COL = "hr"
OUT_CSV = "D:/ESILV_S2/Intern/build_mimic/mimiciv/output/mimic4_column_stats.csv"

PARQUET_SQL = f"read_parquet('{PARQUET_PATH}')"

con = duckdb.connect()
con.execute("PRAGMA threads=8")

## 1. Schema & column classification

In [ ]:
# Inspect the schema and classify each column as numeric, categorical, or datetime
schema = con.execute(f"DESCRIBE SELECT * FROM {PARQUET_SQL}").fetchdf()

def classify_type(duckdb_type: str) -> str:
    t = duckdb_type.upper()
    if t in ("BIGINT", "INTEGER", "DOUBLE", "HUGEINT", "FLOAT", "DECIMAL"):
        return "numeric"
    if "TIMESTAMP" in t or "DATE" in t:
        return "datetime"
    return "categorical"

schema["group"] = schema["column_type"].apply(classify_type)
print(schema["group"].value_counts())
schema

## 2. Table-level overview

In [ ]:
# Table-level overview: row count, patient/admission/ICU-stay counts, hr range
overview = con.execute(f"""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT {SUBJECT_COL}) AS n_subjects,
        COUNT(DISTINCT {HADM_COL}) AS n_admissions,
        COUNT(DISTINCT {STAY_COL}) AS n_icu_stays,
        MIN({HR_COL}) AS hr_min,
        MAX({HR_COL}) AS hr_max
    FROM {PARQUET_SQL}
""").fetchdf()
overview

## 3. Numeric column stats — function + single-column test

In [ ]:
# Define a function that computes summary stats for a numeric column
def numeric_stats(con, parquet_sql, col):
    q = f"""
        SELECT
            '{col}' AS column_name,
            COUNT(*) AS row_count,
            COUNT("{col}") AS non_null_count,
            COUNT(*) - COUNT("{col}") AS null_count,
            ROUND(100.0 * (COUNT(*) - COUNT("{col}")) / COUNT(*), 2) AS null_pct,
            COUNT(DISTINCT "{col}") AS n_distinct,
            MIN("{col}") AS min,
            MAX("{col}") AS max,
            AVG(CAST("{col}" AS DOUBLE)) AS mean,
            MEDIAN("{col}") AS median,
            STDDEV(CAST("{col}" AS DOUBLE)) AS stddev,
            approx_quantile("{col}", 0.01) AS p1,
            approx_quantile("{col}", 0.05) AS p5,
            approx_quantile("{col}", 0.25) AS p25,
            approx_quantile("{col}", 0.75) AS p75,
            approx_quantile("{col}", 0.95) AS p95,
            approx_quantile("{col}", 0.99) AS p99,
            SUM(CASE WHEN "{col}" = 0 THEN 1 ELSE 0 END) AS n_zeros,
            SUM(CASE WHEN "{col}" < 0 THEN 1 ELSE 0 END) AS n_negative
        FROM {parquet_sql}
    """
    return con.execute(q).fetchdf()

In [ ]:
# Quick test: run numeric_stats on a single column
numeric_stats(con, PARQUET_SQL, "heart_rate")

## 4. Categorical column stats — function + single-column test

In [ ]:
# Define a function that computes summary stats for a categorical (string) column
def categorical_stats(con, parquet_sql, col, top_n=5):
    base = con.execute(f"""
        SELECT
            '{col}' AS column_name,
            COUNT(*) AS row_count,
            COUNT("{col}") AS non_null_count,
            COUNT(*) - COUNT("{col}") AS null_count,
            ROUND(100.0 * (COUNT(*) - COUNT("{col}")) / COUNT(*), 2) AS null_pct,
            COUNT(DISTINCT "{col}") AS n_distinct
        FROM {parquet_sql}
    """).fetchdf()

    top_values = con.execute(f"""
        SELECT "{col}" AS value, COUNT(*) AS count
        FROM {parquet_sql}
        WHERE "{col}" IS NOT NULL
        GROUP BY "{col}"
        ORDER BY count DESC
        LIMIT {top_n}
    """).fetchdf()

    base["top_values"] = [list(top_values.itertuples(index=False, name=None))]
    return base

In [ ]:
# Quick test: run categorical_stats on a single column
categorical_stats(con, PARQUET_SQL, "gender")

## 5. Datetime column stats — function + single-column test

In [ ]:
# Define a function that computes summary stats for a datetime column
def datetime_stats(con, parquet_sql, col):
    return con.execute(f"""
        SELECT
            '{col}' AS column_name,
            COUNT(*) AS row_count,
            COUNT("{col}") AS non_null_count,
            COUNT(*) - COUNT("{col}") AS null_count,
            ROUND(100.0 * (COUNT(*) - COUNT("{col}")) / COUNT(*), 2) AS null_pct,
            MIN("{col}") AS min,
            MAX("{col}") AS max
        FROM {parquet_sql}
    """).fetchdf()

In [ ]:
# Quick test: run datetime_stats on a single column
datetime_stats(con, PARQUET_SQL, "charttime_floor")

## 6. Run for all columns

In [ ]:
# Run the matching stats function for every column and combine into one table
results = []
for _, row in schema.iterrows():
    col, group = row["column_name"], row["group"]
    if group == "numeric":
        results.append(numeric_stats(con, PARQUET_SQL, col))
    elif group == "datetime":
        results.append(datetime_stats(con, PARQUET_SQL, col))
    else:
        results.append(categorical_stats(con, PARQUET_SQL, col))

summary = pd.concat(results, ignore_index=True)
summary = summary.merge(schema[["column_name", "column_type", "group"]], on="column_name", how="left")
summary

## 7. Per-ICU-stay coverage

For each column, how many distinct ICU stays have at least one non-null value (vs. the row-level non_null_count above).

In [ ]:
# For each column, count distinct ICU stays that have at least one non-null value
total_stays = con.execute(f"SELECT COUNT(DISTINCT {STAY_COL}) FROM {PARQUET_SQL}").fetchone()[0]
print("total ICU stays:", total_stays)

cols_to_check = [c for c in schema["column_name"] if c != STAY_COL]

exprs = ",\n    ".join(
    f'COUNT(DISTINCT CASE WHEN "{c}" IS NOT NULL THEN {STAY_COL} END) AS "{c}"'
    for c in cols_to_check
)
coverage_wide = con.execute(f"SELECT\n    {exprs}\nFROM {PARQUET_SQL}").fetchdf()

coverage = coverage_wide.T.reset_index()
coverage.columns = ["column_name", "stays_with_value"]
coverage["stay_coverage_pct"] = (100 * coverage["stays_with_value"] / total_stays).round(2)

summary = summary.merge(coverage, on="column_name", how="left")
summary

## 8. Export

In [ ]:
# Save the final per-column stats table to CSV
summary.to_csv(OUT_CSV, index=False)
print(f"Saved {len(summary)} rows -> {OUT_CSV}")

## 9. Sepsis timeline (3h bins, first 14 days)

For each 3-hour bin of ICU stay time (`hr`, starting at 0), count the number of distinct ICU stays present in that bin ("Number of patients") vs. the number of distinct ICU stays with `SepsisLabel = 1` in that bin ("Number of sepsis patients").

In [ ]:
# Bin hr into 3-hour buckets and count distinct ICU stays / subjects per bin, with/without sepsis
import matplotlib.pyplot as plt

BIN_HOURS = 3
MAX_HOURS = 336  # first 14 days

timeline = con.execute(f"""
    SELECT
        CAST(FLOOR({HR_COL} / {BIN_HOURS}) * {BIN_HOURS} AS INTEGER) AS hr_bin,
        COUNT(DISTINCT {STAY_COL}) AS n_patients,
        COUNT(DISTINCT {SUBJECT_COL}) AS n_subjects,
        COUNT(DISTINCT CASE WHEN SepsisLabel = 1 THEN {STAY_COL} END) AS n_sepsis_patients
    FROM {PARQUET_SQL}
    WHERE {HR_COL} >= 0 AND {HR_COL} < {MAX_HOURS}
    GROUP BY hr_bin
    ORDER BY hr_bin
""").fetchdf()

plt.figure(figsize=(14, 5))
plt.plot(timeline["hr_bin"], timeline["n_patients"], label="Number of patients (ICU stays)")
plt.plot(timeline["hr_bin"], timeline["n_subjects"], label="Number of subjects (unique patients)")
plt.plot(timeline["hr_bin"], timeline["n_sepsis_patients"], label="Number of sepsis patients")
plt.xlabel("ICU stay time (hours, 3h bins)")
plt.ylabel("Number of patients")
plt.xticks(range(0, MAX_HOURS + 1, 24))
plt.title("Patients vs Sepsis Patients over ICU stay time")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()